In [49]:
import os
import pandas as pd
import pymssql
from sqlalchemy import create_engine, text
import numpy as np
# from auxiliares import *
from dotenv import load_dotenv

load_dotenv()

True

In [50]:
# variables de entorno para conexión a la BDD
usuario = os.getenv('DB_OLIVOS_ANALITICA_250_USER')
contrasena = os.getenv('DB_OLIVOS_ANALITICA_250_PASS')
host = os.getenv('DB_OLIVOS_ANALITICA_250_HOST')
puerto = 3306
base_de_datos = os.getenv('DB_OLIVOS_ANALITICA_250_NAME')

##deficiones necesarias para subir los datos a la BDD
engine = create_engine(f'mysql+pymysql://{usuario}:{contrasena}@{host}:{puerto}/{base_de_datos}')

conn = pymssql.connect(server= os.getenv('DB_OLIVOS_6_HOST'), 
                       user= os.getenv('DB_OLIVOS_6_USER'),
                       password = os.getenv('DB_OLIVOS_6_PASS'))

In [51]:
# ─── PARÁMETRO: ventana temporal para cancelados ────────────────────────────
# Solo se cargan cancelaciones ocurridas DESDE esta fecha.
# Los contratos ACTIVOS siempre se incluyen completos (su ANTIGUEDAD_ANOS
# captura la lealtad histórica aunque lleven 12+ años en la cartera).
#
# Razonamiento: cancelaciones de hace más de 5 años corresponden a un
# contexto de negocio diferente y pueden introducir ruido en el modelo.
#
# ► Ajustar ANOS_VENTANA según criterio del negocio (recomendado: 5).
# ► Para reentrenamientos futuros la ventana se actualiza sola (fecha relativa).

from datetime import datetime, timedelta

ANOS_VENTANA = 2
fecha_corte = (datetime.now() - timedelta(days=ANOS_VENTANA * 365)).strftime('%Y-%m-%d')

print(f"Ventana temporal activa  : {ANOS_VENTANA} años")
print(f"Fecha de corte           : {fecha_corte}")
print(f"Cancelados incluidos     : desde {fecha_corte} hasta hoy")
print(f"Activos incluidos        : TODOS (sin límite de antigüedad)")

Ventana temporal activa  : 2 años
Fecha de corte           : 2024-05-28
Cancelados incluidos     : desde 2024-05-28 hasta hoy
Activos incluidos        : TODOS (sin límite de antigüedad)


In [57]:
consulta = f"""

SELECT 
    ps.descripcion AS CANAL,
    pg.tercero AS [Nit entidad],
    cp.identificacion AS CEDULA,
    cp.poliza_grupal AS GRUPAL,
    pg.descripcion AS Entidad,
    pp.descripcion AS Producto,
    cp.contrato AS CONTRATO,
    cp.forma_pago AS COD_FORMA_PAGO,
    cp.renovacion AS ESTADO_C,
    cp.finaliza_vigencia AS RENOVACION,
    t.genero,
    pmp.descripcion AS MECANISMO_PAGO,
    ISNULL(DATEDIFF(YEAR, t.nacimiento, GETDATE()), 0) AS EDAD_TT,
    m.descripcion AS MUNICIPIO,
    cp.fecha_afiliacion AS FECHA_EXPEDICION_CONTRATO,

    cpcn_final.FECHA_CANCELACION,
    cpcn_final.MOTIVO_CANCELACION,

    CAST(
        ROUND(
            CASE 
                WHEN cp.renovacion = 'C'
                    THEN DATEDIFF(DAY, cp.fecha_afiliacion, cpcn_final.FECHA_CANCELACION) / 365.25
                ELSE
                    DATEDIFF(DAY, cp.fecha_afiliacion, GETDATE()) / 365.25
            END
        , 2)
    AS DECIMAL(10,2)) AS ANTIGUEDAD_ANOS,

    -- =========================
    -- PERSONAS PROTEGIDAS
    -- =========================
    (
        SELECT COUNT(cpap.contrato_prevision_asegurado)
        FROM contratos_prevision_asegurados cpap
        WHERE cpap.contrato = cp.contrato
          AND (cp.renovacion = 'C' OR cpap.fecha_retiro IS NULL)
    ) AS TOTAL_PERSONAS_PROTEGIDOS,

    pe.PROTEGIDOS_MENORES_65,
    pe.PROTEGIDOS_MAYORES_65,

    -- =========================
    -- MASCOTAS
    -- =========================
    (
        SELECT COUNT(cpaa.contrato_prevision_animal)
        FROM contratos_prevision_animales cpaa
        WHERE cpaa.contrato = cp.contrato
          AND (cp.renovacion = 'C' OR cpaa.fecha_retiro IS NULL)
    ) AS TOTAL_MASCOTAS,

    -- =========================
    -- EXEQUIAS PERSONAS
    -- =========================
    (
        SELECT COUNT(c.id)
        FROM contratos_prevision_amparos c
        JOIN prevision_productos_coberturas d ON d.id = c.id_cobertura
        WHERE c.contrato = cp.contrato
          AND d.prevision_cobertura_id IN ('1','16')
          AND (cp.renovacion = 'C' OR c.fecha_retiro IS NULL)
    ) AS EXEQUIAS_PERSONAS,

    -- =========================
    -- AUXILIOS PERSONAS
    -- =========================
    (
       SELECT COUNT(c.id)
        FROM contratos_prevision_amparos c
        JOIN prevision_productos_coberturas d ON d.id = c.id_cobertura
        WHERE c.contrato = cp.contrato
          AND d.prevision_cobertura_id IN ('3','2','17')
          AND (cp.renovacion = 'C' OR c.fecha_retiro IS NULL)
    ) AS AUXILIOS_PERSONAS,

    -- =========================
    -- ASISTENCIAS PERSONAS
    -- =========================
    (
        SELECT COUNT(c.id)
        FROM contratos_prevision_amparos c
        JOIN prevision_productos_coberturas d ON d.id = c.id_cobertura
        WHERE c.contrato = cp.contrato
          AND d.prevision_cobertura_id IN ('18','26','27','28','29','30','31','32','33','34','35','37','8','9','10','11','20')
          AND (cp.renovacion = 'C' OR c.fecha_retiro IS NULL)
    ) AS ASISTENCIAS_PERSONAS,

    -- =========================
    -- EXEQUIAS MASCOTAS
    -- =========================
    (
        SELECT COUNT(e.contrato_prevision_animal)
        FROM contratos_prevision_animales_amparos e
        JOIN prevision_productos_coberturas d ON d.id = e.id_cobertura
        WHERE e.contrato = cp.contrato
          AND d.prevision_cobertura_id IN ('22','52','53','54','57')
          AND (cp.renovacion = 'C' OR e.fecha_retiro IS NULL)
    ) AS EXEQUIAS_MASCOTAS,

    -- =========================
    -- ASISTENCIAS MASCOTAS
    -- =========================
    (
        SELECT COUNT(e.contrato_prevision_animal)
        FROM contratos_prevision_animales_amparos e
        JOIN prevision_productos_coberturas d ON d.id = e.id_cobertura
        WHERE e.contrato = cp.contrato
          AND d.prevision_cobertura_id IN ('23','24','25','36','38','39','40','41','42','43','44','45','46','47','48','49','50','56','51')
          AND (cp.renovacion = 'C' OR e.fecha_retiro IS NULL)
    ) AS ASISTENCIAS_MASCOTAS,

    -- =========================
-- PERSONAS FALLECIDAS
-- =========================
(
    SELECT COUNT(*)
    FROM contratos_prevision_asegurados cpap
    WHERE cpap.contrato = cp.contrato
      AND cpap.fecha_fallecio IS NOT NULL
) AS TOTAL_PERSONAS_FALLECIDAS,

-- =========================
-- MASCOTAS FALLECIDAS
-- =========================
(
    SELECT COUNT(cpaa.contrato_prevision_animal)
    FROM contratos_prevision_animales cpaa
    WHERE cpaa.contrato = cp.contrato
      AND cpaa.fecha_fallecio IS NOT NULL
) AS TOTAL_MASCOTAS_FALLECIDAS,

    -- =========================
    -- PRIMAS PERSONAS
    -- =========================
    ISNULL((
        SELECT SUM(cpamp.prima)
        FROM contratos_prevision_amparos cpamp
        WHERE cpamp.contrato = cp.contrato
          AND (cp.renovacion = 'C' OR cpamp.fecha_retiro IS NULL)
    ),0) AS TOTAL_PRIMA_PERSONAS,

    -- =========================
    -- PRIMAS MASCOTAS
    -- =========================
    ISNULL((
        SELECT SUM(cpamm.prima)
        FROM contratos_prevision_animales_amparos cpamm
        WHERE cpamm.contrato = cp.contrato
          AND (cp.renovacion = 'C' OR cpamm.fecha_retiro IS NULL)
    ),0) AS TOTAL_PRIMA_MASCOTAS,

    CASE 
        WHEN cp.renovacion = 'C' THEN 'Cancelado'
        ELSE 'Activo'
    END AS ESTADO_ACTUAL

FROM contratos_prevision cp

LEFT JOIN terceros t ON cp.identificacion = t.tercero
LEFT JOIN municipios m ON m.municipio = t.municipio
LEFT JOIN prevision_mecanismos_pago pmp ON pmp.mecanismo_pago = cp.mecanismo_pago
LEFT JOIN prevision_productos pp ON pp.producto_prevision = cp.producto_prevision
INNER JOIN prevision_grupales pg ON pg.poliza_grupal = cp.poliza_grupal
INNER JOIN prevision_segmentos ps ON ps.segmento = pg.segmento

OUTER APPLY (
    SELECT TOP 1
        CAST(cpcn.fecha AS DATE) AS FECHA_CANCELACION,
        ptnm.descripcion AS MOTIVO_CANCELACION
    FROM contratos_prevision_cambio_novedades cpcn
    LEFT JOIN prevision_tipos_novedad_motivos ptnm
        ON ptnm.motivo = cpcn.motivo
    WHERE cpcn.contrato = cp.contrato
      AND cpcn.tipo_novedad = 'C'
    ORDER BY cpcn.fecha DESC
) cpcn_final
-- =========================================================
-- NUEVO JOIN:
-- Se agrega tabla derivada "pe" para obtener:
--   - PROTEGIDOS_MENORES
--   - PROTEGIDOS_MAYORES
--
-- REGLAS:
--   ✔ 1 fila por contrato
--   ✔ excluye titulares (adicional <> 'T')
--   ✔ solo protegidos vigentes
--   ✔ evita duplicidad de contratos
-- =========================================================

LEFT JOIN (

    SELECT
        cp.contrato,

        -- =========================
        -- PROTEGIDOS <= 65 AÑOS
        -- =========================
        SUM(
            CASE 
                WHEN DATEDIFF(YEAR, cpa.fecha_nacimiento, GETDATE()) -
                     CASE 
                         WHEN DATEADD(
                                YEAR,
                                DATEDIFF(YEAR, cpa.fecha_nacimiento, GETDATE()),
                                cpa.fecha_nacimiento
                              ) > GETDATE()
                         THEN 1
                         ELSE 0
                     END <= 65
                THEN 1
                ELSE 0
            END
        ) AS PROTEGIDOS_MENORES_65,

        -- =========================
        -- PROTEGIDOS > 65 AÑOS
        -- =========================
        SUM(
            CASE 
                WHEN DATEDIFF(YEAR, cpa.fecha_nacimiento, GETDATE()) -
                     CASE 
                         WHEN DATEADD(
                                YEAR,
                                DATEDIFF(YEAR, cpa.fecha_nacimiento, GETDATE()),
                                cpa.fecha_nacimiento
                              ) > GETDATE()
                         THEN 1
                         ELSE 0
                     END > 65
                THEN 1
                ELSE 0
            END
        ) AS PROTEGIDOS_MAYORES_65

    FROM contratos_prevision_asegurados cpa

    INNER JOIN contratos_prevision cp
        ON cp.contrato = cpa.contrato

    WHERE 
        -- ✔ Excluir titulares
        cpa.adicional <> 'T'

        -- ✔ Solo protegidos vigentes
        AND (
            cpa.fecha_retiro IS NULL
            OR cpa.fecha_retiro > GETDATE()
        )

    -- ✔ Garantiza 1 fila por contrato
    GROUP BY cp.contrato

) pe
    ON pe.contrato = cp.contrato


WHERE
    (cp.renovacion = 'C' OR cp.renovacion <> 'C')
    AND pg.descripcion NOT LIKE '%prueba%'
	AND (
    cp.renovacion <> 'C'
    OR cpcn_final.FECHA_CANCELACION >= '{fecha_corte}'
)


"""

df = pd.read_sql(consulta, conn)
print(f"Registros cargados: {len(df):,}")
print(f"  Activos   : {(df['ESTADO_ACTUAL']=='Activo').sum():,}")
print(f"  Cancelados: {(df['ESTADO_ACTUAL']=='Cancelado').sum():,}  (cancelados desde {fecha_corte})")
df

C:\Users\asalcedo\AppData\Local\Temp\ipykernel_13488\3423431992.py:273: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(consulta, conn)


Registros cargados: 448,856
  Activos   : 314,728
  Cancelados: 134,128  (cancelados desde 2024-05-28)


,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,EXEQUIAS_PERSONAS,AUXILIOS_PERSONAS,ASISTENCIAS_PERSONAS,EXEQUIAS_MASCOTAS,ASISTENCIAS_MASCOTAS,TOTAL_PERSONAS_FALLECIDAS,TOTAL_MASCOTAS_FALLECIDAS,TOTAL_PRIMA_PERSONAS,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL
0,Sedes Alternas,800108302-06,70100976,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00000212,30,R,2026-08-15,...,11,1,2,0,0,1,0,52050.0,0.0,Activo
1,Sedes Alternas,800108302-06,32532070,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00030545,360,R,2026-08-04,...,3,1,2,0,0,0,0,20200.0,0.0,Activo
2,Asociadas,890981395,3628971,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048177,360,R,2026-06-28,...,7,0,0,2,0,0,1,18900.0,13000.0,Activo
3,Asociadas,890981395,43678255,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048226,360,R,2026-10-13,...,2,1,0,0,0,0,0,9500.0,0.0,Activo
4,Asociadas,890981395,71875111,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00039494,360,R,2026-10-26,...,2,1,0,0,0,0,0,9500.0,0.0,Activo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448851,Asociadas,890904902,43758829,92,PIO XII,PIO XII - PFI - No asociados,BA-08833,180,A,2027-05-25,...,4,2,1,0,0,0,0,22100.0,0.0,Activo
448852,Asociadas,890905859,1038814846,83,COOTRAMED,Cootramed - PFI,10592913,180,A,2026-08-26,...,2,2,1,0,0,0,0,16600.0,0.0,Activo
448853,Asociadas,890904902,71706339,92,PIO XII,PIO XII - PFI - No asociados,BA-08772,360,A,2027-04-27,...,3,1,1,0,0,0,0,17600.0,0.0,Activo
448854,Corporativo,890913990,1001477563,522,JIRO S A S,Jiro - PFI Individual,10570305,30,C,2026-05-01,...,2,1,1,0,0,0,0,7450.0,0.0,Cancelado


In [ ]:
# ## filtrar por cedula para pruebas
# df = df[df['CEDULA'] == '43493268']
# df

,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,EXEQUIAS_PERSONAS,AUXILIOS_PERSONAS,ASISTENCIAS_PERSONAS,EXEQUIAS_MASCOTAS,ASISTENCIAS_MASCOTAS,TOTAL_PERSONAS_FALLECIDAS,TOTAL_MASCOTAS_FALLECIDAS,TOTAL_PRIMA_PERSONAS,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL
3,Sedes Alternas,800108302-06,43493268,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI SOS Plus - Menor 60,00000005,30,R,2026-06-13,...,18,1,3,0,0,5,3,124490.0,0.0,Activo


In [58]:
#Validar duplicados
print(f"Registros duplicados: {df.duplicated(subset=['CONTRATO']).sum()}")

Registros duplicados: 0


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448856 entries, 0 to 448855
Data columns (total 32 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   CANAL                      448856 non-null  object        
 1   Nit entidad                448856 non-null  object        
 2   CEDULA                     448544 non-null  object        
 3   GRUPAL                     448856 non-null  object        
 4   Entidad                    448856 non-null  object        
 5   Producto                   448856 non-null  object        
 6   CONTRATO                   448856 non-null  object        
 7   COD_FORMA_PAGO             448856 non-null  int64         
 8   ESTADO_C                   448856 non-null  object        
 9   RENOVACION                 448856 non-null  datetime64[ns]
 10  genero                     306227 non-null  object        
 11  MECANISMO_PAGO             448396 non-null  object  

In [60]:
# filtrar por una cedula
df_registro = df[df['CEDULA'] == '1017182437']
df_registro


,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,EXEQUIAS_PERSONAS,AUXILIOS_PERSONAS,ASISTENCIAS_PERSONAS,EXEQUIAS_MASCOTAS,ASISTENCIAS_MASCOTAS,TOTAL_PERSONAS_FALLECIDAS,TOTAL_MASCOTAS_FALLECIDAS,TOTAL_PRIMA_PERSONAS,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL
112686,Asociadas,890922036,1017182437,653,TIPALMA SAS TITULARES- CONFIAR,Tipalma - Prevision Exequial,00416515,30,C,2026-07-31,...,1,0,0,0,0,0,0,2350.0,0.0,Cancelado
195988,Asociadas,890922036,1017182437,653,TIPALMA SAS TITULARES- CONFIAR,Tipalma - Prevision Exequial,00474085,30,C,2026-08-01,...,1,0,0,0,0,0,0,2350.0,0.0,Cancelado
199944,Asociadas,890922036,1017182437,653,TIPALMA SAS TITULARES- CONFIAR,Tipalma - Prevision Exequial,00482506,30,A,2026-08-01,...,1,0,0,0,0,0,0,2350.0,0.0,Activo


In [61]:
# 
df['PFI'] = df['Producto'].str.contains('PFI', case=False, na=False).astype(int)

df = df[~df['CEDULA'].str.contains('TR', case=False, na=False)].copy()

df = df[(df['CANAL'] != 'Intermediarios de Previsión')].copy()

conn.close()

print(len(df))

df

444816


,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,AUXILIOS_PERSONAS,ASISTENCIAS_PERSONAS,EXEQUIAS_MASCOTAS,ASISTENCIAS_MASCOTAS,TOTAL_PERSONAS_FALLECIDAS,TOTAL_MASCOTAS_FALLECIDAS,TOTAL_PRIMA_PERSONAS,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL,PFI
0,Sedes Alternas,800108302-06,70100976,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00000212,30,R,2026-08-15,...,1,2,0,0,1,0,52050.0,0.0,Activo,1
1,Sedes Alternas,800108302-06,32532070,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00030545,360,R,2026-08-04,...,1,2,0,0,0,0,20200.0,0.0,Activo,1
2,Asociadas,890981395,3628971,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048177,360,R,2026-06-28,...,0,0,2,0,0,1,18900.0,13000.0,Activo,0
3,Asociadas,890981395,43678255,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048226,360,R,2026-10-13,...,1,0,0,0,0,0,9500.0,0.0,Activo,0
4,Asociadas,890981395,71875111,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00039494,360,R,2026-10-26,...,1,0,0,0,0,0,9500.0,0.0,Activo,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448851,Asociadas,890904902,43758829,92,PIO XII,PIO XII - PFI - No asociados,BA-08833,180,A,2027-05-25,...,2,1,0,0,0,0,22100.0,0.0,Activo,1
448852,Asociadas,890905859,1038814846,83,COOTRAMED,Cootramed - PFI,10592913,180,A,2026-08-26,...,2,1,0,0,0,0,16600.0,0.0,Activo,1
448853,Asociadas,890904902,71706339,92,PIO XII,PIO XII - PFI - No asociados,BA-08772,360,A,2027-04-27,...,1,1,0,0,0,0,17600.0,0.0,Activo,1
448854,Corporativo,890913990,1001477563,522,JIRO S A S,Jiro - PFI Individual,10570305,30,C,2026-05-01,...,1,1,0,0,0,0,7450.0,0.0,Cancelado,1


In [62]:
def tiene_pfi(valor):
    if valor == 1:
        return 'SI'
    else:
        return 'NO'

In [64]:
df['TOTAL_PRIMA'] = df['TOTAL_PRIMA_PERSONAS'] + df['TOTAL_PRIMA_MASCOTAS']

df['VALOR_RENOVACION'] = (df['TOTAL_PRIMA'] * df['COD_FORMA_PAGO']) / 30

df['VALOR_RENOVACION'] = df['VALOR_RENOVACION'].round(1)

df['TOTAL_TITULARES'] = 1

df['TOTAL_PERSONAS'] = df['TOTAL_PERSONAS_PROTEGIDOS'] - df['TOTAL_TITULARES']

df['TIENE PFI ACTUALMENTE?'] = df['PFI'].apply(tiene_pfi)

df['fecha_mes'] = pd.to_datetime('2026-05-01')

df

,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,TOTAL_PRIMA_PERSONAS,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL,PFI,TOTAL_PRIMA,VALOR_RENOVACION,TOTAL_TITULARES,TOTAL_PERSONAS,TIENE PFI ACTUALMENTE?,fecha_mes
0,Sedes Alternas,800108302-06,70100976,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00000212,30,R,2026-08-15,...,52050.0,0.0,Activo,1,52050.0,52050.0,1,10,SI,2026-05-01
1,Sedes Alternas,800108302-06,32532070,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00030545,360,R,2026-08-04,...,20200.0,0.0,Activo,1,20200.0,242400.0,1,2,SI,2026-05-01
2,Asociadas,890981395,3628971,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048177,360,R,2026-06-28,...,18900.0,13000.0,Activo,0,31900.0,382800.0,1,6,NO,2026-05-01
3,Asociadas,890981395,43678255,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048226,360,R,2026-10-13,...,9500.0,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01
4,Asociadas,890981395,71875111,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00039494,360,R,2026-10-26,...,9500.0,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448851,Asociadas,890904902,43758829,92,PIO XII,PIO XII - PFI - No asociados,BA-08833,180,A,2027-05-25,...,22100.0,0.0,Activo,1,22100.0,132600.0,1,3,SI,2026-05-01
448852,Asociadas,890905859,1038814846,83,COOTRAMED,Cootramed - PFI,10592913,180,A,2026-08-26,...,16600.0,0.0,Activo,1,16600.0,99600.0,1,1,SI,2026-05-01
448853,Asociadas,890904902,71706339,92,PIO XII,PIO XII - PFI - No asociados,BA-08772,360,A,2027-04-27,...,17600.0,0.0,Activo,1,17600.0,211200.0,1,2,SI,2026-05-01
448854,Corporativo,890913990,1001477563,522,JIRO S A S,Jiro - PFI Individual,10570305,30,C,2026-05-01,...,7450.0,0.0,Cancelado,1,7450.0,7450.0,1,1,SI,2026-05-01


In [65]:
df_contactabilidad = pd.read_parquet(
    r'C:\Users\asalcedo\Documents\Procesos Analista de informacion\dimensionamiento_org\data\contactabilidad.parquet')
df_contactabilidad

,Cedula_contacto,Telefono_contacto,WhatsApp,llamada telefonica,dia_semana,Rango_hora
0,0,6044802790,0.003,0.997,Miércoles,10 a 12 m
1,00,3167402456,0.000,1.000,Martes,1 a 3 pm
2,000,3015154060,0.011,0.989,Jueves,10 a 12 m
3,0000,3187387888,0.000,1.000,Martes,10 a 12 m
4,00000,3016612249,0.014,0.986,Lunes,10 a 12 m
...,...,...,...,...,...,...
166088,9993954,3154240478,0.000,1.000,Miércoles,7 a 9 am
166089,9994103,3146321126,0.125,0.875,Martes,7 a 9 am
166090,9994116,3135106550,0.000,1.000,Lunes,7 a 9 am
166091,9994861,3103641461,0.000,1.000,Jueves,10 a 12 m


In [66]:
df_contactabilidad = df_contactabilidad.rename(columns={'Telefono_contacto': 'Telefono mas efectivo',
    'llamada telefonica': 'porcentaje llamada telefonica', 'WhatsApp': 'porcentaje WhatsApp', 
    'dia_semana': 'dia mas efectivo', 'Rango_hora': 'Rango hora mas efectivo',
    'Cedula_contacto': 'CEDULA'})
df_contactabilidad

,CEDULA,Telefono mas efectivo,porcentaje WhatsApp,porcentaje llamada telefonica,dia mas efectivo,Rango hora mas efectivo
0,0,6044802790,0.003,0.997,Miércoles,10 a 12 m
1,00,3167402456,0.000,1.000,Martes,1 a 3 pm
2,000,3015154060,0.011,0.989,Jueves,10 a 12 m
3,0000,3187387888,0.000,1.000,Martes,10 a 12 m
4,00000,3016612249,0.014,0.986,Lunes,10 a 12 m
...,...,...,...,...,...,...
166088,9993954,3154240478,0.000,1.000,Miércoles,7 a 9 am
166089,9994103,3146321126,0.125,0.875,Martes,7 a 9 am
166090,9994116,3135106550,0.000,1.000,Lunes,7 a 9 am
166091,9994861,3103641461,0.000,1.000,Jueves,10 a 12 m


In [67]:
df_base_train = df.merge(df_contactabilidad[['CEDULA','Telefono mas efectivo']], 
    on='CEDULA', how='left')

df_base_train['Telefono mas efectivo'] = df_base_train['Telefono mas efectivo'].fillna('0')

df_base_train['Telefono mas efectivo'] = df_base_train['Telefono mas efectivo'].astype(str)

df_base_train

,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL,PFI,TOTAL_PRIMA,VALOR_RENOVACION,TOTAL_TITULARES,TOTAL_PERSONAS,TIENE PFI ACTUALMENTE?,fecha_mes,Telefono mas efectivo
0,Sedes Alternas,800108302-06,70100976,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00000212,30,R,2026-08-15,...,0.0,Activo,1,52050.0,52050.0,1,10,SI,2026-05-01,3147349612
1,Sedes Alternas,800108302-06,32532070,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00030545,360,R,2026-08-04,...,0.0,Activo,1,20200.0,242400.0,1,2,SI,2026-05-01,0
2,Asociadas,890981395,3628971,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048177,360,R,2026-06-28,...,13000.0,Activo,0,31900.0,382800.0,1,6,NO,2026-05-01,6044877063
3,Asociadas,890981395,43678255,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048226,360,R,2026-10-13,...,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01,3113493755
4,Asociadas,890981395,71875111,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00039494,360,R,2026-10-26,...,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01,3136647444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444811,Asociadas,890904902,43758829,92,PIO XII,PIO XII - PFI - No asociados,BA-08833,180,A,2027-05-25,...,0.0,Activo,1,22100.0,132600.0,1,3,SI,2026-05-01,0
444812,Asociadas,890905859,1038814846,83,COOTRAMED,Cootramed - PFI,10592913,180,A,2026-08-26,...,0.0,Activo,1,16600.0,99600.0,1,1,SI,2026-05-01,0
444813,Asociadas,890904902,71706339,92,PIO XII,PIO XII - PFI - No asociados,BA-08772,360,A,2027-04-27,...,0.0,Activo,1,17600.0,211200.0,1,2,SI,2026-05-01,3214333028
444814,Corporativo,890913990,1001477563,522,JIRO S A S,Jiro - PFI Individual,10570305,30,C,2026-05-01,...,0.0,Cancelado,1,7450.0,7450.0,1,1,SI,2026-05-01,0


In [68]:
df_base_train.info()
df_base_train

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444816 entries, 0 to 444815
Data columns (total 40 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   CANAL                      444816 non-null  object        
 1   Nit entidad                444816 non-null  object        
 2   CEDULA                     444504 non-null  object        
 3   GRUPAL                     444816 non-null  object        
 4   Entidad                    444816 non-null  object        
 5   Producto                   444816 non-null  object        
 6   CONTRATO                   444816 non-null  object        
 7   COD_FORMA_PAGO             444816 non-null  int64         
 8   ESTADO_C                   444816 non-null  object        
 9   RENOVACION                 444816 non-null  datetime64[ns]
 10  genero                     302214 non-null  object        
 11  MECANISMO_PAGO             444356 non-null  object  

,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL,PFI,TOTAL_PRIMA,VALOR_RENOVACION,TOTAL_TITULARES,TOTAL_PERSONAS,TIENE PFI ACTUALMENTE?,fecha_mes,Telefono mas efectivo
0,Sedes Alternas,800108302-06,70100976,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00000212,30,R,2026-08-15,...,0.0,Activo,1,52050.0,52050.0,1,10,SI,2026-05-01,3147349612
1,Sedes Alternas,800108302-06,32532070,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00030545,360,R,2026-08-04,...,0.0,Activo,1,20200.0,242400.0,1,2,SI,2026-05-01,0
2,Asociadas,890981395,3628971,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048177,360,R,2026-06-28,...,13000.0,Activo,0,31900.0,382800.0,1,6,NO,2026-05-01,6044877063
3,Asociadas,890981395,43678255,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048226,360,R,2026-10-13,...,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01,3113493755
4,Asociadas,890981395,71875111,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00039494,360,R,2026-10-26,...,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01,3136647444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444811,Asociadas,890904902,43758829,92,PIO XII,PIO XII - PFI - No asociados,BA-08833,180,A,2027-05-25,...,0.0,Activo,1,22100.0,132600.0,1,3,SI,2026-05-01,0
444812,Asociadas,890905859,1038814846,83,COOTRAMED,Cootramed - PFI,10592913,180,A,2026-08-26,...,0.0,Activo,1,16600.0,99600.0,1,1,SI,2026-05-01,0
444813,Asociadas,890904902,71706339,92,PIO XII,PIO XII - PFI - No asociados,BA-08772,360,A,2027-04-27,...,0.0,Activo,1,17600.0,211200.0,1,2,SI,2026-05-01,3214333028
444814,Corporativo,890913990,1001477563,522,JIRO S A S,Jiro - PFI Individual,10570305,30,C,2026-05-01,...,0.0,Cancelado,1,7450.0,7450.0,1,1,SI,2026-05-01,0


In [71]:
consolidado = df_base_train[
    [
        "CANAL",
        "Nit entidad",
        "CEDULA",
        "GRUPAL",
        "Entidad",
        "Producto",
        "CONTRATO",
        "COD_FORMA_PAGO",
        "ESTADO_C",
        "RENOVACION",
        "genero",
        "MECANISMO_PAGO",
        "EDAD_TT",
        "MUNICIPIO",
        "FECHA_EXPEDICION_CONTRATO",
        "FECHA_CANCELACION",
        "MOTIVO_CANCELACION",
        "ANTIGUEDAD_ANOS",
        "TOTAL_PERSONAS_PROTEGIDOS",
        "PROTEGIDOS_MENORES_65",
        "PROTEGIDOS_MAYORES_65",
        "TOTAL_MASCOTAS",
        "EXEQUIAS_PERSONAS",
        "AUXILIOS_PERSONAS",
        "ASISTENCIAS_PERSONAS",
        "EXEQUIAS_MASCOTAS",
        "ASISTENCIAS_MASCOTAS",
        "TOTAL_PERSONAS_FALLECIDAS",
        "TOTAL_MASCOTAS_FALLECIDAS",
        "TOTAL_PRIMA_PERSONAS",
        "TOTAL_PRIMA_MASCOTAS",
        "ESTADO_ACTUAL",
        "PFI",
        "TOTAL_PRIMA",
        "VALOR_RENOVACION",
        "TOTAL_TITULARES",
        "TOTAL_PERSONAS",
        "TIENE PFI ACTUALMENTE?",
        "fecha_mes",
        "Telefono mas efectivo"
    ]
].copy()
consolidado

,CANAL,Nit entidad,CEDULA,GRUPAL,Entidad,Producto,CONTRATO,COD_FORMA_PAGO,ESTADO_C,RENOVACION,...,TOTAL_PRIMA_MASCOTAS,ESTADO_ACTUAL,PFI,TOTAL_PRIMA,VALOR_RENOVACION,TOTAL_TITULARES,TOTAL_PERSONAS,TIENE PFI ACTUALMENTE?,fecha_mes,Telefono mas efectivo
0,Sedes Alternas,800108302-06,70100976,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00000212,30,R,2026-08-15,...,0.0,Activo,1,52050.0,52050.0,1,10,SI,2026-05-01,3147349612
1,Sedes Alternas,800108302-06,32532070,326,INDIVIDUALES MEDELLIN,Individuales Medellín - PFI PLUS,00030545,360,R,2026-08-04,...,0.0,Activo,1,20200.0,242400.0,1,2,SI,2026-05-01,0
2,Asociadas,890981395,3628971,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048177,360,R,2026-06-28,...,13000.0,Activo,0,31900.0,382800.0,1,6,NO,2026-05-01,6044877063
3,Asociadas,890981395,43678255,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00048226,360,R,2026-10-13,...,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01,3113493755
4,Asociadas,890981395,71875111,12,CONFIAR COOPERATIVA FINANCIERA,Confiar - Previsión,00039494,360,R,2026-10-26,...,0.0,Activo,0,9500.0,114000.0,1,2,NO,2026-05-01,3136647444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444811,Asociadas,890904902,43758829,92,PIO XII,PIO XII - PFI - No asociados,BA-08833,180,A,2027-05-25,...,0.0,Activo,1,22100.0,132600.0,1,3,SI,2026-05-01,0
444812,Asociadas,890905859,1038814846,83,COOTRAMED,Cootramed - PFI,10592913,180,A,2026-08-26,...,0.0,Activo,1,16600.0,99600.0,1,1,SI,2026-05-01,0
444813,Asociadas,890904902,71706339,92,PIO XII,PIO XII - PFI - No asociados,BA-08772,360,A,2027-04-27,...,0.0,Activo,1,17600.0,211200.0,1,2,SI,2026-05-01,3214333028
444814,Corporativo,890913990,1001477563,522,JIRO S A S,Jiro - PFI Individual,10570305,30,C,2026-05-01,...,0.0,Cancelado,1,7450.0,7450.0,1,1,SI,2026-05-01,0


In [72]:
consolidado.to_parquet('../data/raw/df_consolidado.parquet')